In [3]:
# ==================== 第一步：修复路径问题 ====================
import sys
from pathlib import Path
import argparse 
# 自动找到项目根目录
current_dir = Path.cwd()  # 当前目录
project_root = current_dir.parent  # 上一级目录就是项目根目录

# 把项目根目录加入Python搜索路径
sys.path.insert(0, str(project_root))

print("✅ 路径设置完成")
print(f"   当前目录: {current_dir.name}")
print(f"   项目根目录: {project_root}")

# ==================== 第二步：导入所有需要的库 ====================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 导入项目模块（现在应该可以找到了）
from config.config import config
from data.data_collector import DataCollector
from data.data_processor import DataProcessor

# 设置绘图样式
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ 所有库导入成功！")

# ============== 项目内模块导入（若失败，给出清晰提示） ==============
try:
    from config.config import config
except Exception as e:
    print("[ERROR] 无法导入 config.config.config，请确认项目结构与 PYTHONPATH。")
    raise

try:
    from data.data_collector import DataCollector
except Exception as e:
    print("[ERROR] 无法导入 data.data_collector.DataCollector。")
    print("        请确认文件存在且类名/构造函数签名匹配。")
    raise

try:
    from data.data_processor import DataProcessor
except Exception as e:
    print("[ERROR] 无法导入 data.data_processor.DataProcessor。")
    raise

try:
    from utils.helpers import save_pickle
except Exception as e:
    print("[ERROR] 无法导入 utils.helpers.save_pickle。")
    raise


# ============== 绘图辅助：统一风格（仅 matplotlib） ==============
def ensure_outdir(path: Path):
    path.mkdir(parents=True, exist_ok=True)


def plot_price_and_volume(df: pd.DataFrame, symbol: str, outdir: Path):
    """
    df 需要包含 index 为日期、列包含 ['close', 'volume']。
    """
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    axes[0].plot(df.index, df["close"], linewidth=2, label="Close")
    axes[0].set_title(f"{symbol} 价格走势")
    axes[0].set_xlabel("日期")
    axes[0].set_ylabel("价格 ($)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].bar(df.index, df["volume"], alpha=0.7, label="Volume")
    axes[1].set_title(f"{symbol} 成交量")
    axes[1].set_xlabel("日期")
    axes[1].set_ylabel("成交量")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    outfile = outdir / f"{symbol}_price_volume.png"
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"[图已保存] {outfile}")


def plot_indicators(df: pd.DataFrame, symbol: str, outdir: Path):
    """
    需要列：close, sma_short, sma_long, rsi, macd, macd_signal, macd_diff
    """
    fig, axes = plt.subplots(3, 1, figsize=(12, 10))

    # 收盘价 + 均线
    axes[0].plot(df.index, df["close"], linewidth=2, label="Close")
    if "sma_short" in df.columns:
        axes[0].plot(df.index, df["sma_short"], linestyle="--", label="SMA(20)")
    if "sma_long" in df.columns:
        axes[0].plot(df.index, df["sma_long"], linestyle="--", label="SMA(50)")
    axes[0].set_title(f"{symbol} 价格与移动平均线")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # RSI
    if "rsi" in df.columns:
        axes[1].plot(df.index, df["rsi"], linewidth=2, label="RSI")
        axes[1].axhline(y=70, linestyle="--", alpha=0.7, label="超买(70)")
        axes[1].axhline(y=30, linestyle="--", alpha=0.7, label="超卖(30)")
        axes[1].set_title("RSI 指标")
        axes[1].set_ylim([0, 100])
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
    else:
        axes[1].text(0.5, 0.5, "无 RSI 列", ha="center", va="center")
        axes[1].axis("off")

    # MACD
    if {"macd", "macd_signal", "macd_diff"}.issubset(df.columns):
        axes[2].plot(df.index, df["macd"], linewidth=2, label="MACD")
        axes[2].plot(df.index, df["macd_signal"], linewidth=2, label="Signal")
        axes[2].bar(df.index, df["macd_diff"], alpha=0.3, label="Histogram")
        axes[2].set_title("MACD 指标")
        axes[2].legend()
        axes[2].grid(True, alpha=0.3)
    else:
        axes[2].text(0.5, 0.5, "无 MACD 相关列", ha="center", va="center")
        axes[2].axis("off")

    plt.tight_layout()
    outfile = outdir / f"{symbol}_indicators.png"
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"[图已保存] {outfile}")


def plot_sentiment_pie(sentiment: dict, symbol: str, outdir: Path):
    """
    需要键：positive_mentions / neutral_mentions / negative_mentions
    """
    pos = sentiment.get("positive_mentions", 0)
    neu = sentiment.get("neutral_mentions", 0)
    neg = sentiment.get("negative_mentions", 0)
    values = [pos, neu, neg]
    labels = ["正面", "中性", "负面"]

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.pie(values, labels=labels, autopct="%1.1f%%", startangle=90)
    ax.set_title(f"{symbol} 社交媒体情绪分布")
    outfile = outdir / f"{symbol}_sentiment_pie.png"
    plt.savefig(outfile, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"[图已保存] {outfile}")


# ============== 主流程 ==============
def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--symbols", nargs="+", default=["AAPL", "MSFT", "GOOGL"], help="股票代码列表")
    parser.add_argument("--start", default="2023-01-01", help="开始日期 YYYY-MM-DD")
    parser.add_argument("--end", default="2024-01-01", help="结束日期 YYYY-MM-DD")
    parser.add_argument("--outdir", default="outputs", help="图片与中间结果输出目录")
    args = parser.parse_args()

    SYMBOLS = args.symbols
    START_DATE = args.start
    END_DATE = args.end
    OUTDIR = Path(args.outdir)
    ensure_outdir(OUTDIR)

    print(f"股票列表: {SYMBOLS}")
    print(f"时间范围: {START_DATE} 到 {END_DATE}")

    # 1) 初始化收集器/处理器
    collector = DataCollector(SYMBOLS)
    processor = DataProcessor()

    # 2) 收集第一只股票的价格数据（示例）
    symbol = SYMBOLS[0]
    print(f"\n正在收集 {symbol} 的数据 ...")
    price_data = collector.collect_stock_data(symbol, START_DATE, END_DATE)

    if not isinstance(price_data.index, pd.DatetimeIndex):
        # 尽量保证时间索引
        try:
            price_data = price_data.copy()
            price_data.index = pd.to_datetime(price_data.index)
        except Exception:
            pass

    print(f"\n数据形状: {price_data.shape}")
    print("\n前5行数据:")
    print(price_data.head())

    # 可视化价格与成交量
    plot_price_and_volume(price_data, symbol, OUTDIR)

    # 3) 计算技术指标并可视化
    print("\n正在计算技术指标 ...")
    price_with_ind = processor.calculate_technical_indicators(price_data)
    added_cols = len(price_with_ind.columns) - len(price_data.columns)
    print(f"\n添加的指标数量: {added_cols}")
    print("\n所有列名:")
    print(price_with_ind.columns.tolist())
    plot_indicators(price_with_ind, symbol, OUTDIR)

    # 4) 财务数据
    print("\n正在收集财务数据 ...")
    financial_data = collector.collect_financial_data(symbol)
    print("财务数据（摘要）:")
    for k, v in financial_data.items():
        if k != "symbol":
            print(f"  {k}: {v}")

    # 5) 新闻与情绪
    print("\n正在收集新闻数据 ...")
    news_data = collector.collect_news_data(symbol, days=7)
    print(f"新闻数量: {len(news_data)}")
    print("\n最新新闻（最多3条）:")
    for i, news in enumerate(news_data[:3], 1):
        title = news.get("title", "")
        desc = news.get("description", "")
        sent = news.get("sentiment", "")
        print(f"\n{i}. {title}\n   {desc}\n   情绪: {sent}")

    print("\n正在收集情绪数据 ...")
    sentiment = collector.collect_sentiment_data(symbol)
    print("市场情绪数据:")
    for k in ["sentiment_score", "sentiment_label", "positive_mentions",
              "neutral_mentions", "negative_mentions", "total_mentions"]:
        if k in sentiment:
            print(f"  {k}: {sentiment[k]}")
    # 可视化情绪
    plot_sentiment_pie(sentiment, symbol, OUTDIR)

    # 6) 技术分析摘要
    print("\n正在生成技术分析摘要 ...")
    tech_summary = processor.generate_technical_summary(price_with_ind)
    try:
        import json
        print(json.dumps(tech_summary, indent=2, ensure_ascii=False))
    except Exception:
        print(tech_summary)

    # 7) 收集完整数据集并对价格数据再次加工
    print("\n正在收集完整数据集 ...")
    complete_data = collector.collect_all_data(symbol, START_DATE, END_DATE)
    if "price_data" in complete_data:
        complete_data["price_data"] = processor.calculate_technical_indicators(
            complete_data["price_data"]
        )

    print("\n完整数据集包含：")
    pd_len = len(complete_data.get("price_data", []))
    fd_len = len(complete_data.get("financial_data", {}))
    nd_len = len(complete_data.get("news_data", []))
    print(f"  - 价格数据: {pd_len} 天")
    print(f"  - 财务数据: {fd_len} 个指标")
    print(f"  - 新闻数据: {nd_len} 条新闻")
    print("  - 情绪数据: ✓")

    # 8) 保存数据（使用项目内的 save_pickle 与 config.PROCESSED_DATA_DIR）
    out_price_pkl = Path(config.PROCESSED_DATA_DIR) / f"{symbol}_price_data.pkl"
    out_complete_pkl = Path(config.PROCESSED_DATA_DIR) / f"{symbol}_complete_data.pkl"
    ensure_outdir(out_price_pkl.parent)

    save_pickle(complete_data["price_data"], out_price_pkl)
    save_pickle(complete_data, out_complete_pkl)
    print(f"\n✅ 数据已保存：\n  - {out_price_pkl}\n  - {out_complete_pkl}")

    print(f"\n所有图片已输出到：{OUTDIR.resolve()}")


if __name__ == "__main__":
    main()

✅ 路径设置完成
   当前目录: notebooks
   项目根目录: /Users/liuxinwei/trading_agents_project
✅ 所有库导入成功！


usage: ipykernel_launcher.py [-h] [--symbols SYMBOLS [SYMBOLS ...]]
                             [--start START] [--end END] [--outdir OUTDIR]
ipykernel_launcher.py: error: unrecognized arguments: -f /Users/liuxinwei/Library/Jupyter/runtime/kernel-721feda8-cfa5-46fc-ac03-613ec248a89e.json


SystemExit: 2